<a href="https://colab.research.google.com/github/corebankingfiap-dev/core_banking/blob/main/01_Data_Lake_Bancos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bancos

Bronze - Dado Bruto

Silver - Dado Limpo e Padronizado

Gold - Tranformado e Calculado

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from google.colab import drive
from datetime import datetime

# 1. CONEXÃO E CONFIGURAÇÃO DE CAMINHOS
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

segmento = "bancos"
tickers = ["BBAS3.SA", "ITUB4.SA", "BBDC4.SA", "SANB11.SA", "BPAC11.SA"]
BASE_PATH = '/content/drive/MyDrive/DATA_LAKE/'

# Garantir que toda a estrutura de pastas existe
for pasta in ['01_Bronze', '02_Silver', '03_Gold']:
    os.makedirs(os.path.join(BASE_PATH, pasta), exist_ok=True)

print(f"🚀 Iniciando Pipeline Completo: {segmento.upper()}")

# --- ETAPA 1: BRONZE (Extração Bruta Diária) ---
# Puxa o dado bruto e salva sem mexer em nada
df_bruto = yf.download(tickers, period="5y", interval="1d", auto_adjust=True)['Close']
df_bruto.to_csv(f'{BASE_PATH}01_Bronze/b3_{segmento}_bruto.csv')
print("✅ Camada BRONZE: Dados brutos salvos.")

# --- ETAPA 2: SILVER (Limpeza e Formatação Visual) ---
# 1. Tratamento técnico (ffill e remoção de .SA)
df_limpo = df_bruto.ffill().sort_index()
df_limpo = df_limpo[~df_limpo.index.duplicated(keep='first')]
df_limpo.columns = [col.replace('.SA', '') for col in df_limpo.columns]

# 2. Salvamento Técnico (para o Python ler)
df_limpo.to_csv(f'{BASE_PATH}02_Silver/b3_{segmento}_limpo.csv')

# 3. Criação do arquivo FORMATADO (R$) para você abrir e ler
def fmt_money(x): return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
df_view = df_limpo.map(fmt_money)
df_view.to_csv(f'{BASE_PATH}02_Silver/b3_{segmento}_FORMATADO.csv')
print("✅ Camada SILVER: Dados limpos e arquivo VIEW (R$) gerados.")

# --- ETAPA 3: GOLD (Processamento Quant e Inteligência) ---
# A. Cálculo de Log-Retornos
log_retornos = np.log(df_limpo / df_limpo.shift(1)).dropna()

# B. Métricas Anualizadas (252 dias úteis)
retorno_anual = log_retornos.mean() * 252
vol_anual = log_retornos.std() * np.sqrt(252)
sharpe_ratio = retorno_anual / vol_anual

# C. Tabela de Decisão (Gold)
df_gold = pd.DataFrame({
    'Retorno Anualizado (%)': (retorno_anual * 100).round(2),
    'Volatilidade Anual (%)': (vol_anual * 100).round(2),
    'Indice Sharpe': sharpe_ratio.round(2)
}).sort_values(by='Indice Sharpe', ascending=False)

# D. Salvamento da Inteligência
df_gold.to_csv(f'{BASE_PATH}03_Gold/b3_{segmento}_analise.csv')
print("✅ Camada GOLD: Métricas de Sharpe e Volatilidade finalizadas.")

# --- SAÍDA FINAL NA TELA ---
print(f"\n--- RELATÓRIO FINAL: {segmento.upper()} ---")
print("\n🏆 RANKING DE EFICIÊNCIA (GOLD):")
display(df_gold)

print("\n💰 ÚLTIMOS PREÇOS FORMATADOS (SILVER):")
display(df_view.tail(3))

[                       0%                       ]

🚀 Iniciando Pipeline Completo: BANCOS


[*********************100%***********************]  5 of 5 completed

✅ Camada BRONZE: Dados brutos salvos.
✅ Camada SILVER: Dados limpos e arquivo VIEW (R$) gerados.
✅ Camada GOLD: Métricas de Sharpe e Volatilidade finalizadas.

--- RELATÓRIO FINAL: BANCOS ---

🏆 RANKING DE EFICIÊNCIA (GOLD):


,Retorno Anualizado (%),Volatilidade Anual (%),Indice Sharpe
ITUB4,17.93,25.19,0.71
BBAS3,17.60,26.74,0.66
BPAC11,20.06,35.19,0.57
BBDC4,5.77,29.99,0.19
SANB11,2.81,26.61,0.11



💰 ÚLTIMOS PREÇOS FORMATADOS (SILVER):


,BBAS3,BBDC4,BPAC11,ITUB4,SANB11
Date,,,,,
2026-03-10,"R$ 25,14","R$ 20,03","R$ 57,92","R$ 43,80","R$ 32,25"
2026-03-11,"R$ 25,34","R$ 19,94","R$ 58,22","R$ 43,89","R$ 32,00"
2026-03-12,"R$ 24,23","R$ 19,39","R$ 56,10","R$ 42,69","R$ 30,58"
